In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import *
from pyspark.sql.window import *

spark = SparkSession.builder.appName("UserAnalysis").getOrCreate()

# Create df_accounts
df_accounts = spark.createDataFrame(
    [
        ("U001", "2023-01-01", "New York"),
        ("U002", "2023-01-05", "Chicago"),
        ("U003", "2023-01-10", "San Francisco"),
    ],
    ["user_id", "account_created_date", "location"],
)

# Create df_activities
df_activities = spark.createDataFrame(
    [
        ("U001", "2023-02-01", "Login"),
        ("U001", "2023-02-01", "Login"),
        ("U002", "2023-02-05", "File Upload"),
        ("U002", "2023-02-05", "File Upload"),
        ("U003", "2023-02-10", "Logout"),
    ],
    ["user_id", "activity_date", "activity_type"],
)

# Create df_exit_surveys
df_exit_surveys = spark.createDataFrame(
    [
        ("U001", "2023-03-01", "Moved to a competitor"),
        ("U002", "2023-03-05", "Not user-friendly"),
        ("U003", "2023-03-10", "High pricing"),
    ],
    ["user_id", "exit_date", "exit_reason"],
)

In [5]:
window_spec = Window.partitionBy("user_id").orderBy(desc("activity_date"))

df_act_rn = df_activities.withColumn("rn", row_number().over(window_spec)).filter(
    col("rn") == 1
)

In [12]:
df_accounts.join(df_act_rn, "user_id", "inner").join(
    df_exit_surveys, "user_id", "inner"
).select(
    "account_created_date",
    "activity_date",
    "activity_type",
    "exit_date",
    "exit_reason",
    "location",
    "user_id",
).show()

+--------------------+-------------+-------------+----------+--------------------+-------------+-------+
|account_created_date|activity_date|activity_type| exit_date|         exit_reason|     location|user_id|
+--------------------+-------------+-------------+----------+--------------------+-------------+-------+
|          2023-01-01|   2023-02-01|        Login|2023-03-01|Moved to a compet...|     New York|   U001|
|          2023-01-05|   2023-02-05|  File Upload|2023-03-05|   Not user-friendly|      Chicago|   U002|
|          2023-01-10|   2023-02-10|       Logout|2023-03-10|        High pricing|San Francisco|   U003|
+--------------------+-------------+-------------+----------+--------------------+-------------+-------+

